# COMP1859 Information Retrieval Coursework: SmartEdU Academic IR System

Section markers refer to the brief's marking scheme:
- **Part A:** Design and Development (A1 architecture, A2 retrieval methods, A3 structured vs unstructured data, A4 functionality demonstration)
- **Part B:** Implementation and Evaluation (not started)
- **Part C:** Screencast (not started)
- **Part D:** Clarity and Organisation (report)

---
## Part A1: System Architecture
**To do.** The architecture diagram and design justification will be written once the structured/unstructured join and ranked retrieval are built, so that it shows the finished system.

Components built so far:
- Unstructured side: PDF to text conversion, in-memory corpus, preprocessing, inverted index, Boolean keyword search
- Structured side: SQLite database (`students`, `courses`, `enrollments`, `attendance`, `documents_metadata`)

---
## Data Preparation (setup for A2 and A3)
These cells must run first because preprocessing and indexing depend on the loaded text.
### Step 1: Convert PDFs to text

In [11]:
import os  # walk folders and build paths
from pypdf import PdfReader  # pure-python pdf reader

ROOT = "data/raw_documents"  # folder that holds the pdfs
converted = []  # plain list of files done
failed = []  # plain list of files that errored

for folder, _, files in os.walk(ROOT):  # visit every subfolder
    for name in files:  # each file in the folder
        if not name.lower().endswith(".pdf"):  # skip non-pdf files
            continue  # go to next file
        pdf = os.path.join(folder, name)  # full pdf path
        txt = pdf[:-4] + ".txt"  # same name with .txt extension
        try:  # catch unreadable pdfs
            reader = PdfReader(pdf)  # open the pdf
            text = "\n".join(page.extract_text() or "" for page in reader.pages)  # join all pages
            with open(txt, "w", encoding="utf-8") as f:  # create the text file
                f.write(text)  # save the text
            converted.append(txt)  # record success
        except Exception as e:  # something went wrong
            failed.append((pdf, str(e)))  # record the failure

print(len(converted), "converted,", len(failed), "failed")  # summary

18 converted, 0 failed


### Step 2: Load the text corpus into memory
All converted `.txt` files are loaded into a dictionary keyed by relative path (`doc_id`).

In [12]:
from pathlib import Path

ROOT = Path("data/raw_documents")

def load_corpus(root):
    corpus = {}
    for path in sorted(root.rglob("*.txt")):
        doc_id = str(path.relative_to(root))  # e.g. "cs101/assignment1.txt"
        corpus[doc_id] = path.read_text(encoding="utf-8", errors="replace")
    return corpus

docs = load_corpus(ROOT)
print(len(docs), "documents loaded")

28 documents loaded


---
## Part A2: Retrieval Methods
### Text preprocessing
Pipeline shared by indexing and querying: lowercasing, punctuation removal, stopword removal and Porter stemming.

In [13]:
import nltk
nltk.download("punkt")  # download the tokenizer models
nltk.download("stopwords")  # download the stopword list
import re

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\dungs\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\dungs\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\dungs\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [14]:
def preprocess_text(text):
    #lowercase text
    text = text.lower()
    #remove punctuation
    text = re.sub(r'[^\w\s]', '', text)
    #remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    #remove stopwords
    stopwords = set(nltk.corpus.stopwords.words('english'))
    text = ' '.join(word for word in text.split() if word not in stopwords)
    #apply stemming
    stemmer = nltk.stem.PorterStemmer()
    text = ' '.join(stemmer.stem(word) for word in text.split())
    return text

In [15]:
#load some text to show that processing works
sample_text = docs[next(iter(docs))]  # get the first document
print("Original text:", sample_text[:500], "\n")
print("Processed text:", preprocess_text(sample_text)[:500], "\n")

Original text: Implementing Inverted Index for Academic Information Retrieval
Student: STU005 | Course: CRS01
This report details the implementation of an inverted index data structure
for academic document retrieval. An inverted index maps vocabulary terms to
postings lists containing document IDs and term frequencies.
We benchmarked document tokenization, Porter stemming, and stopword
filtering across an educational corpus. Stopword removal re 

Processed text: implement invert index academ inform retriev student stu005 cours crs01 report detail implement invert index data structur academ document retriev invert index map vocabulari term post list contain document id term frequenc benchmark document token porter stem stopword filter across educ corpu stopword remov reduc dictionari size 38 percent maintain vocabulari coverag domainspecif queri queri process evalu use boolean conjunct disjunct time complex post list intersect optim use skip pointer resu 



### 1. Inverted index and Boolean keyword search
Builds the inverted index (term to set of documents) and a Boolean AND keyword search over it. Results are unranked.

In [16]:
inverted_index = {}

#build the inverted index
for doc_id, text in docs.items():
    processed_text = preprocess_text(text)
    for word in processed_text.split():
        if word not in inverted_index:
            inverted_index[word] = set()
        inverted_index[word].add(doc_id)

#function to search the inverted index
def search(query):
    processed_query = preprocess_text(query)
    query_words = processed_query.split()
    if not query_words:
        return set()  # return empty set for empty query
    result_sets = [inverted_index.get(word, set()) for word in query_words]
    if not result_sets:
        return set()  # return empty set if no words found
    result_docs = set.intersection(*result_sets)
    return result_docs

### Sample keyword search
Output of this cell is a candidate screenshot for Part A4.

In [17]:
#sample search
search_query = "information retrieval"
search_results = search(search_query)
print(f"Search results for '{search_query}':")

for doc_id in search_results:
    print(f"Content: {docs[doc_id]}...")  # print first 200 characters of the document
    print(doc_id)

Search results for 'information retrieval':
Content: Title: Course Evaluation Feedback: Lecture Pacing and Lab Sessions for COMP1859
Student ID: STU005
Course ID: CRS01

Overall the Information Retrieval lectures by Dr. Ayodeji Ibitoye are very inspiring and clear. The practical labs on inverted index construction and BM25 ranking gave strong hands-on intuition. However, more guidance on vector embeddings and neural IR search models would be greatly appreciated for the final coursework prototype....
feedback\DOC019_Feedback_COMP1859_Pacing.txt
Content: Implementing Inverted Index for Academic Information Retrieval
Student: STU005 | Course: CRS01
This report details the implementation of an inverted index data structure
for academic document retrieval. An inverted index maps vocabulary terms to
postings lists containing document IDs and term frequencies.
We benchmarked document tokenization, Porter stemming, and stopword
filtering across an educational corpus. Stopword removal reduced d

### 2. BM25 ranked retrieval
Scores each document for a query using term frequency, inverse document frequency and document length normalisation, so results are ranked by relevance instead of returned as an unordered set.

In [18]:
import math
from collections import Counter

# preprocess every document once and reuse the result (preprocess_text returns a string, so split it)
doc_tokens = {d: preprocess_text(t).split() for d, t in docs.items()}
N = len(doc_tokens)                                                   # number of documents
avgdl = sum(len(t) for t in doc_tokens.values()) / N                  # average document length
tf = {d: Counter(t) for d, t in doc_tokens.items()}                   # term frequency per document
df = Counter(term for t in doc_tokens.values() for term in set(t))    # number of documents containing each term

def idf(term):
    # "+ 1" keeps the value positive even for terms that appear in most documents
    return math.log((N - df[term] + 0.5) / (df[term] + 0.5) + 1)

def bm25_search(query, k1=1.5, b=0.75, top_k=10):
    terms = preprocess_text(query).split()          # same preprocessing as the documents
    scores = {}
    for d, tokens in doc_tokens.items():
        dl = len(tokens)                            # length of this document
        score = 0.0
        for t in terms:
            f = tf[d].get(t, 0)                     # how often the term appears here
            if f:
                score += idf(t) * f * (k1 + 1) / (f + k1 * (1 - b + b * dl / avgdl))
        if score > 0:
            scores[d] = score
    return sorted(scores.items(), key=lambda x: -x[1])[:top_k]

In [19]:
def show(query):
    boolean = sorted(search(query))
    ranked = bm25_search(query)
    print(f"Query: '{query}'")
    print(f"  Boolean AND: {len(boolean)} result(s)")
    for d in boolean:
        print("   ", d)
    print(f"  BM25: {len(ranked)} result(s)")
    for d, s in ranked:
        print(f"   {s:.3f}  {d}")
    print()

show("information retrieval")
show("BM25 ranking embeddings")

Query: 'information retrieval'
  Boolean AND: 4 result(s)
    assignments\DOC001_Assignment_Inverted_Index.txt
    feedback\DOC019_Feedback_COMP1859_Pacing.txt
    feedback\DOC026_Email_Thesis_Supervision_STU005.txt
    syllabi\DOC013_Syllabus_COMP1859_Information_Retrieval.txt
  BM25: 5 result(s)
   5.058  syllabi\DOC013_Syllabus_COMP1859_Information_Retrieval.txt
   4.692  feedback\DOC026_Email_Thesis_Supervision_STU005.txt
   3.873  assignments\DOC001_Assignment_Inverted_Index.txt
   3.764  feedback\DOC019_Feedback_COMP1859_Pacing.txt
   1.519  assignments\DOC005_Assignment_DB_Normalization.txt

Query: 'BM25 ranking embeddings'
  Boolean AND: 1 result(s)
    feedback\DOC019_Feedback_COMP1859_Pacing.txt
  BM25: 4 result(s)
   7.409  feedback\DOC019_Feedback_COMP1859_Pacing.txt
   5.340  assignments\DOC002_Assignment_TFIDF_vs_BM25.txt
   3.508  syllabi\DOC013_Syllabus_COMP1859_Information_Retrieval.txt
   1.779  assignments\DOC010_Assignment_Sorting_Profiling.txt



### Observations
- For "information retrieval", Boolean AND returned 4 documents. BM25 returned the same 4 plus one weaker match (DOC005, score 1.519), ranked last.
- For "BM25 ranking embeddings", Boolean AND returned only DOC019, the single document containing all three terms. BM25 also returned DOC002 (TF-IDF vs BM25 assignment), which is topically relevant but excluded by AND because it lacks one of the terms.
- AND gives strict matches but can miss relevant documents; BM25 recovers them and orders by score, at the cost of some low-scoring partial matches.

---
## Part A3: Structured vs Unstructured Data
The unstructured side (PDF text, preprocessing, inverted index) is handled by the data preparation and A2 cells above. The cells below cover the structured side.
**To do:** join `documents_metadata` to the student and course tables to integrate the two sides.
### Structured data: SQLite database
Connects to the supplied database and lists its tables and row counts.

In [20]:
import sqlite3  # built-in library for SQLite databases
import pandas as pd  # prints query results as tables

DB_PATH = "data/structured/smartedu.db"  # the supplied database file

con = sqlite3.connect(DB_PATH)  # open a connection to the file

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con)  # ask the database which tables it holds
print(tables)  # show the table names

for name in tables["name"]:  # check every table
    count = pd.read_sql(f"SELECT COUNT(*) AS n FROM {name}", con)["n"][0]  # number of rows
    print(name, count, "rows")  # e.g. students 70 rows

print(pd.read_sql("SELECT * FROM students LIMIT 3", con))  # peek at a few rows

                 name
0            students
1             courses
2         enrollments
3     sqlite_sequence
4          attendance
5  documents_metadata
students 70 rows
courses 6 rows
enrollments 246 rows
sqlite_sequence 2 rows
attendance 246 rows
documents_metadata 28 rows
  student_id first_name last_name                          email  \
0     STU001        Mia     Smith      mia.smith1@smartedu.ac.uk   
1     STU002       Owen  Williams  owen.williams2@smartedu.ac.uk   
2     STU003       Emma  Williams  emma.williams3@smartedu.ac.uk   

                department  enrollment_year   gpa  
0             Data Science             2023  3.32  
1  Artificial Intelligence             2022  3.92  
2     Software Engineering             2023  3.10  


### Structured schema inspection
Column names, types and primary keys of the `students` table.

In [21]:
print(pd.read_sql("PRAGMA table_info(students)", con))  # one row per column: name, type, whether it is a key

   cid             name     type  notnull dflt_value  pk
0    0       student_id     TEXT        0       None   1
1    1       first_name     TEXT        1       None   0
2    2        last_name     TEXT        1       None   0
3    3            email     TEXT        1       None   0
4    4       department     TEXT        1       None   0
5    5  enrollment_year  INTEGER        1       None   0
6    6              gpa     REAL        1       None   0


### Linking documents to student and course records
The corpus keys are file paths, while `documents_metadata` identifies documents by number (e.g. `DOC001`). The document number is extracted from each key and used to join the text documents to the student, course, enrolment and attendance tables.

In [22]:
def doc_number(key):
    m = re.search(r"DOC\d+", key)  # "assignments\DOC001_Assignment_Inverted_Index.txt" -> "DOC001"
    return m.group() if m else None

key_to_id = {k: doc_number(k) for k in docs}  # corpus key -> metadata doc_id

meta_ids = set(pd.read_sql("SELECT doc_id FROM documents_metadata", con)["doc_id"])
unmatched = [k for k, v in key_to_id.items() if v not in meta_ids]
print(len(key_to_id), "corpus documents;", len(unmatched), "without a metadata row")
print(unmatched)

28 corpus documents; 0 without a metadata row
[]


In [23]:
query = """
SELECT m.doc_id, m.title, m.doc_type, m.department AS doc_department,
       s.student_id, s.first_name, s.last_name, s.gpa,
       c.course_id, c.course_code, c.course_name, c.instructor,
       e.semester, e.final_grade, e.letter_grade,
       a.attendance_rate
FROM documents_metadata m
LEFT JOIN students    s ON m.student_id = s.student_id
LEFT JOIN courses     c ON m.course_id  = c.course_id
LEFT JOIN enrollments e ON m.student_id = e.student_id AND m.course_id = e.course_id
LEFT JOIN attendance  a ON m.student_id = a.student_id AND m.course_id = a.course_id
"""
meta = pd.read_sql(query, con)

print(len(meta), "rows;", meta["doc_id"].nunique(), "unique documents")
print(meta.isna().sum()[lambda x: x > 0])  # columns with missing values
meta.head(3)

28 rows; 28 unique documents
student_id          6
first_name          6
last_name           6
gpa                 6
semester           13
final_grade        13
letter_grade       13
attendance_rate    13
dtype: int64


,doc_id,title,doc_type,doc_department,student_id,first_name,last_name,gpa,course_id,course_code,course_name,instructor,semester,final_grade,letter_grade,attendance_rate
0,DOC001,Implementing Inverted Index for Academic Infor...,assignment,Computer Science,STU005,Samuel,Jackson,3.21,CRS01,COMP1859,Information Retrieval,Dr. Ayodeji Ibitoye,Fall 2025,75.4,A,90.0
1,DOC002,Comparative Evaluation of TF-IDF versus BM25 o...,assignment,Computer Science,STU012,Benjamin,Gonzalez,3.12,CRS01,COMP1859,Information Retrieval,Dr. Ayodeji Ibitoye,NaN,NaN,NaN,NaN
2,DOC003,Supervised Learning Models for Student Academi...,assignment,Artificial Intelligence,STU018,Gabriel,Thompson,3.51,CRS02,COMP1788,Machine Learning Systems,Dr. Sarah Jenkins,NaN,NaN,NaN,NaN


In [24]:
meta_by_id = meta.set_index("doc_id")

def with_details(doc_keys):
    rows = meta_by_id.loc[[key_to_id[k] for k in doc_keys]]
    return rows[["title", "doc_type", "student_id", "first_name", "last_name",
                 "course_code", "letter_grade", "attendance_rate"]]

with_details(sorted(search("information retrieval")))

,title,doc_type,student_id,first_name,last_name,course_code,letter_grade,attendance_rate
doc_id,,,,,,,,
DOC001,Implementing Inverted Index for Academic Infor...,assignment,STU005,Samuel,Jackson,COMP1859,A,90.0
DOC019,Course Evaluation Feedback: Lecture Pacing and...,feedback,STU005,Samuel,Jackson,COMP1859,A,90.0
DOC026,Student Email: Inquiry Regarding Master Thesis...,email,STU005,Samuel,Jackson,COMP1859,A,90.0
DOC013,Course Syllabus: COMP1859 Information Retrieval,syllabus,NaN,NaN,NaN,COMP1859,NaN,NaN


In [25]:
enr = pd.read_sql("SELECT student_id, course_id FROM enrollments", con)
orphans = meta[meta["student_id"].notna() & meta["letter_grade"].isna()][["doc_id", "doc_type", "student_id", "course_id"]].copy()
orphans["enrolled_in"] = orphans["student_id"].map(
    lambda s: ", ".join(sorted(enr[enr["student_id"] == s]["course_id"])) or "none")
orphans

,doc_id,doc_type,student_id,course_id,enrolled_in
1,DOC002,assignment,STU012,CRS01,"CRS02, CRS03, CRS05, CRS06"
2,DOC003,assignment,STU018,CRS02,"CRS01, CRS03, CRS05"
5,DOC006,assignment,STU039,CRS03,"CRS02, CRS04, CRS05"
7,DOC008,assignment,STU051,CRS04,"CRS01, CRS02, CRS03, CRS05"
19,DOC020,feedback,STU012,CRS01,"CRS02, CRS03, CRS05, CRS06"
20,DOC021,email,STU018,CRS02,"CRS01, CRS03, CRS05"
23,DOC024,email,STU007,CRS03,"CRS01, CRS02, CRS04, CRS05"


---
## Part A4: Functionality Demonstration
### Search with filters
Runs a BM25 text search, then restricts the results by course, document type or department using the linked metadata. Each result shows its score and the student's details.

In [26]:
def search_documents(query, course_code=None, doc_type=None, department=None, top_k=10):
    ranked = bm25_search(query, top_k=len(docs))   # score every matching document before filtering
    rows = []
    for key, score in ranked:
        r = meta_by_id.loc[key_to_id[key]]         # structured record for this document
        if course_code and r["course_code"] != course_code:
            continue
        if doc_type and r["doc_type"] != doc_type:
            continue
        if department and r["doc_department"] != department:
            continue
        student = "(course document)" if pd.isna(r["first_name"]) else f'{r["first_name"]} {r["last_name"]}'
        rows.append({"doc_id": r.name, "score": round(score, 3), "title": r["title"],
                     "doc_type": r["doc_type"], "course": r["course_code"],
                     "student": student, "grade": r["letter_grade"]})
    return pd.DataFrame(rows).head(top_k)

In [27]:
print("No filter:")
display(search_documents("inverted index"))

print("Feedback only:")
display(search_documents("inverted index", doc_type="feedback"))

print("COMP1859 assignments:")
display(search_documents("information retrieval", course_code="COMP1859", doc_type="assignment"))

No filter:


,doc_id,score,title,doc_type,course,student,grade
0,DOC001,5.578,Implementing Inverted Index for Academic Infor...,assignment,COMP1859,Samuel Jackson,A
1,DOC019,3.854,Course Evaluation Feedback: Lecture Pacing and...,feedback,COMP1859,Samuel Jackson,A
2,DOC013,3.815,Course Syllabus: COMP1859 Information Retrieval,syllabus,COMP1859,(course document),NaN
3,DOC026,1.652,Student Email: Inquiry Regarding Master Thesis...,email,COMP1859,Samuel Jackson,A
4,DOC015,1.508,Course Syllabus: COMP1649 Database Design & Ma...,syllabus,COMP1649,(course document),NaN
5,DOC005,1.367,Relational Database Normalization and B-Tree I...,assignment,COMP1649,Gabriel Williams,A


Feedback only:


,doc_id,score,title,doc_type,course,student,grade
0,DOC019,3.854,Course Evaluation Feedback: Lecture Pacing and...,feedback,COMP1859,Samuel Jackson,A


COMP1859 assignments:


,doc_id,score,title,doc_type,course,student,grade
0,DOC001,3.873,Implementing Inverted Index for Academic Infor...,assignment,COMP1859,Samuel Jackson,A


### For the report

In [28]:
def show_results(df):
    if df.empty:
        print("No results\n")
        return
    out = df.copy()
    out["title"] = out["title"].str.slice(0, 38)
    print(out.to_string(index=False), "\n")

In [29]:
print("Software Engineering documents:")
show_results(search_documents("database", department="Software Engineering"))

print("Filter that matches nothing:")
show_results(search_documents("inverted index", course_code="COMP9999"))

Software Engineering documents:
doc_id  score                                  title   doc_type   course           student grade
DOC015  3.540 Course Syllabus: COMP1649 Database Des   syllabus COMP1649 (course document)   NaN
DOC022  2.399 Student Email: Clarification on Databa      email COMP1649  Gabriel Williams     A
DOC005  1.932 Relational Database Normalization and  assignment COMP1649  Gabriel Williams     A 

Filter that matches nothing:
No results



### Structured retrieval (SQL)
Exact-match queries over the structured tables: attendance, grades and student records. Unlike the text search, these return rows that satisfy a condition, with no relevance score.

In [30]:
def run_sql(query, params=()):
    return pd.read_sql(query, con, params=params)   # "?" placeholders keep user values out of the SQL text

def low_attendance(course_code, threshold=75):
    return run_sql("""
        SELECT s.student_id, s.first_name, s.last_name, a.attendance_rate
        FROM attendance a
        JOIN students s ON a.student_id = s.student_id
        JOIN courses  c ON a.course_id  = c.course_id
        WHERE c.course_code = ? AND a.attendance_rate < ?
        ORDER BY a.attendance_rate
    """, (course_code, threshold))

def course_grades(course_code, min_grade=0):
    return run_sql("""
        SELECT s.student_id, s.first_name, s.last_name, e.semester, e.final_grade, e.letter_grade
        FROM enrollments e
        JOIN students s ON e.student_id = s.student_id
        JOIN courses  c ON e.course_id  = c.course_id
        WHERE c.course_code = ? AND e.final_grade >= ?
        ORDER BY e.final_grade DESC
    """, (course_code, min_grade))

def students_in_department(department, min_gpa=0):
    return run_sql("""
        SELECT student_id, first_name, last_name, enrollment_year, gpa
        FROM students
        WHERE department = ? AND gpa >= ?
        ORDER BY gpa DESC
    """, (department, min_gpa))

def show_table(df):
    print("No results\n" if df.empty else df.to_string(index=False) + "\n")

In [31]:
print("COMP1859 students with attendance below 75%:")
show_table(low_attendance("COMP1859", 75))

print("COMP1859 grades, highest first (top 10):")
show_table(course_grades("COMP1859").head(10))

print("Data Science students with GPA of 3.5 or above:")
show_table(students_in_department("Data Science", 3.5))

COMP1859 students with attendance below 75%:
student_id first_name last_name  attendance_rate
    STU019      Riley  Gonzalez             50.0
    STU030      Emily     Davis             50.0
    STU056     Aubrey   Jackson             50.0
    STU059      James  Anderson             50.0
    STU067     Oliver     Smith             50.0
    STU021     Aubrey  Gonzalez             55.0
    STU043   Benjamin     Davis             55.0
    STU048      Henry  Gonzalez             55.0
    STU050       Lily     Perez             55.0
    STU057     Harper     Perez             55.0
    STU064  Charlotte Rodriguez             55.0
    STU065       Ella    Miller             55.0
    STU008    Jackson    Martin             60.0
    STU014       Liam    Martin             60.0
    STU018    Gabriel  Thompson             60.0
    STU032       Owen     Jones             60.0
    STU034  Alexander  Thompson             60.0
    STU046      James  Williams             60.0
    STU007     Harper He

---
## Part B: Implementation and Evaluation
### Shared evaluation helpers
Precision, recall, average precision, timing and filter helpers used by both the structured (B1) and text (B2) evaluations.

In [32]:
import time

def precision_recall(retrieved, relevant):
    hits = len(set(retrieved) & set(relevant))
    return (hits / len(retrieved) if retrieved else 0.0,
            hits / len(relevant) if relevant else 0.0)

def average_precision(ranked, relevant):          # rewards putting relevant documents early
    hits, total = 0, 0.0
    for i, d in enumerate(ranked, start=1):
        if d in relevant:
            hits += 1
            total += hits / i
    return total / len(relevant) if relevant else 0.0

def avg_ms(fn, repeats=200):                      # average over many runs, one run is too short to time
    start = time.perf_counter()
    for _ in range(repeats):
        fn()
    return (time.perf_counter() - start) / repeats * 1000

def passes(doc_id, filters):
    return all(meta_by_id.loc[doc_id, col] == val for col, val in filters.items())

### Part B1: Structured retrieval accuracy and speed
Each SQL function is checked against an independent computation in pandas, then timed.

In [33]:
students_df = pd.read_sql("SELECT * FROM students", con)
courses_df  = pd.read_sql("SELECT * FROM courses", con)
att_df      = pd.read_sql("SELECT * FROM attendance", con)
enr_df      = pd.read_sql("SELECT * FROM enrollments", con)

def cid(code):
    return courses_df.loc[courses_df["course_code"] == code, "course_id"].iloc[0]

# ground truth computed with pandas, independently of the SQL queries
truth = {
    "S1 attendance < 75, COMP1859": set(att_df[(att_df.course_id == cid("COMP1859")) & (att_df.attendance_rate < 75)].student_id),
    "S2 grade >= 70, COMP1859":     set(enr_df[(enr_df.course_id == cid("COMP1859")) & (enr_df.final_grade >= 70)].student_id),
    "S3 Data Science, GPA >= 3.5":  set(students_df[(students_df.department == "Data Science") & (students_df.gpa >= 3.5)].student_id),
}
runs = {
    "S1 attendance < 75, COMP1859": lambda: low_attendance("COMP1859", 75),
    "S2 grade >= 70, COMP1859":     lambda: course_grades("COMP1859", 70),
    "S3 Data Science, GPA >= 3.5":  lambda: students_in_department("Data Science", 3.5),
}

srows = []
for name, fn in runs.items():
    out = fn()
    srows.append({"query": name, "expected": len(truth[name]), "rows": len(out),
                  "distinct": out["student_id"].nunique(),
                  "matches_truth": set(out["student_id"]) == truth[name],
                  "ms": round(avg_ms(fn, 50), 3)})
print(pd.DataFrame(srows).to_string(index=False))

                       query  expected  rows  distinct  matches_truth    ms
S1 attendance < 75, COMP1859        30    30        30           True 0.328
    S2 grade >= 70, COMP1859        20    20        20           True 0.330
 S3 Data Science, GPA >= 3.5         7     7         7           True 0.286


### Part B2: Test queries and answer key
Each test query has a hand-judged set of relevant documents, based on reading all 28 documents. Filtered queries (F1, F2) list only the relevant documents that also satisfy the filter.

In [34]:
# (id, query, filters, relevant doc_ids)
TEXT_QUERIES = [
    ("Q1",  "inverted index",               {}, {"DOC001", "DOC013", "DOC019"}),
    ("Q2",  "information retrieval",        {}, {"DOC001", "DOC002", "DOC013", "DOC019", "DOC020", "DOC026"}),
    ("Q3",  "BM25 ranking",                 {}, {"DOC002", "DOC013", "DOC019"}),
    ("Q4",  "database normalization",       {}, {"DOC005", "DOC015", "DOC022"}),
    ("Q5",  "student attendance",           {}, {"DOC003", "DOC024"}),
    ("Q6",  "postpone coursework due date", {}, {"DOC021"}),   # deliberate vocabulary mismatch
    ("Q7",  "kubernetes",                   {}, {"DOC011", "DOC018", "DOC025"}),
    ("Q8",  "JWT authentication",           {}, {"DOC008", "DOC016", "DOC027"}),
    ("Q9",  "machine learning",             {}, {"DOC003", "DOC004", "DOC014", "DOC021", "DOC023"}),
    ("Q10", "Dijkstra shortest path",       {}, {"DOC009", "DOC017"}),
    ("F1",  "inverted index",               {"doc_type": "feedback"}, {"DOC019"}),
    ("F2",  "information retrieval",        {"course_code": "COMP1859", "doc_type": "assignment"}, {"DOC001", "DOC002"}),
]

### Part B2: Text retrieval results
Boolean AND vs BM25 on the test queries: precision, recall, average precision (BM25 only) and mean latency. Results are saved to `test_results.csv`.

In [35]:
rows = []
for qid, query, filters, relevant in TEXT_QUERIES:
    bool_ids = [d for d in (key_to_id[k] for k in sorted(search(query))) if passes(d, filters)]
    bm25_ids = [d for d in (key_to_id[k] for k, _ in bm25_search(query, top_k=len(docs))) if passes(d, filters)]
    bp, br = precision_recall(bool_ids, relevant)
    mp, mr = precision_recall(bm25_ids, relevant)
    rows.append({"id": qid, "query": query, "rel": len(relevant),
                 "bool_n": len(bool_ids), "bool_P": bp, "bool_R": br,
                 "bm25_n": len(bm25_ids), "bm25_P": mp, "bm25_R": mr,
                 "bm25_AP": average_precision(bm25_ids, relevant),
                 "bool_ms": avg_ms(lambda: search(query)),
                 "bm25_ms": avg_ms(lambda: bm25_search(query, top_k=len(docs)))})

results = pd.DataFrame(rows).round(3)
print(results.to_string(index=False))
print("\nMean over", len(results), "queries:")
print(results[["bool_P", "bool_R", "bm25_P", "bm25_R", "bm25_AP", "bool_ms", "bm25_ms"]].mean().round(3).to_string())
results.to_csv("test_results.csv", index=False)

 id                        query  rel  bool_n  bool_P  bool_R  bm25_n  bm25_P  bm25_R  bm25_AP  bool_ms  bm25_ms
 Q1               inverted index    3       3     1.0   1.000       6   0.500   1.000    1.000    0.603    0.651
 Q2        information retrieval    6       4     1.0   0.667       5   0.800   0.667    0.667    0.650    0.587
 Q3                 BM25 ranking    3       3     1.0   1.000       4   0.750   1.000    1.000    0.600    0.601
 Q4       database normalization    3       3     1.0   1.000       4   0.750   1.000    1.000    0.588    0.646
 Q5           student attendance    2       2     1.0   1.000      24   0.083   1.000    1.000    0.611    0.674
 Q6 postpone coursework due date    1       0     0.0   0.000       8   0.125   1.000    1.000    0.570    0.584
 Q7                   kubernetes    3       3     1.0   1.000       3   1.000   1.000    1.000    0.541    0.561
 Q8           JWT authentication    3       2     1.0   0.667       3   1.000   1.000    1.000  

### Part B1: Comparison across data types
Summary of structured (SQL) and unstructured (text) retrieval, using the results above.

In [36]:
struct = pd.DataFrame(srows)
print(f"Structured (SQL): {struct['matches_truth'].sum()} of {len(struct)} queries match the independent check; mean {struct['ms'].mean():.3f} ms")
print(f"Boolean text:     mean precision {results['bool_P'].mean():.3f}, mean recall {results['bool_R'].mean():.3f}; mean {results['bool_ms'].mean():.3f} ms")
print(f"BM25 text:        mean precision {results['bm25_P'].mean():.3f}, mean recall {results['bm25_R'].mean():.3f}, MAP {results['bm25_AP'].mean():.3f}; mean {results['bm25_ms'].mean():.3f} ms")

Structured (SQL): 3 of 3 queries match the independent check; mean 0.315 ms
Boolean text:     mean precision 0.917, mean recall 0.744; mean 0.591 ms
BM25 text:        mean precision 0.727, mean recall 0.931, MAP 0.931; mean 0.621 ms
